# Notebook 05 — Modele LSTM i GRU

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych  

---

## Cel notebooka

Trenujemy dwa modele sieci neuronowych rekurencyjnych:

- **LSTM** (Long Short-Term Memory): wychwytuje długookresowe zależności w szeregach czasowych
- **GRU** (Gated Recurrent Unit): lżejsza wersja LSTM, często osiąga porównywalną wydajność

## Kluczowe decyzje projektowe
- **Lookback window = 20 dni**: model widzi ostatnie 20 dni historii przed prognozą
- **Brak data leakage**: skalowanie fit TYLKO na zbiorze train
- **OOF predictions**: predykcje out-of-fold zapisujemy dla Stacking (Notebook 08)
- **Podział:** Train 2019-2021 | Val 2022 | Test 2023-2024

In [1]:
# ============================================================
# Konfiguracja wspolna
# ============================================================
import os, warnings
os.environ.setdefault('KERAS_BACKEND', 'torch')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
warnings.filterwarnings('ignore')
import importlib
import pandas as pd, numpy as np
import pipeline_lib as pl, models_lib as ml, run_analysis as ra
for m in (pl, ml, ra): importlib.reload(m)
pd.set_option('display.width', 160)
print(f'Ziarno losowe: {pl.SEED} | horyzont: {pl.HORIZON} sesji | embargo: {pl.EMBARGO} sesji')


Ziarno losowe: 42 | horyzont: 5 sesji | embargo: 10 sesji


In [2]:
# ============================================================
# Trening modeli bazowych w schemacie purged walk-forward
# ============================================================
# Uwaga: pelny przebieg trwa okolo 30 minut. Jesli prognozy zostaly juz policzone,
# komorka je wczytuje zamiast powtarzac trening.
from pathlib import Path
import run_walkforward as rw

oof_path = Path('data/processed/oof_walkforward.csv')
if not oof_path.exists():
    rw.main()
oof = pd.read_csv(oof_path, index_col=0, parse_dates=True)
folds = pd.read_csv('results/tables/05_walkforward_folds.csv')
print(f'Prognozy pozaprobkowe: {len(oof)} sesji, {oof.index.min().date()} -> {oof.index.max().date()}')
print(f'\nPodzialy walidacyjne ({len(folds)} foldow):')
print(folds.to_string(index=False))


Prognozy pozaprobkowe: 1705 sesji, 2019-10-18 -> 2026-08-14

Podzialy walidacyjne (10 foldow):
 fold train_start  train_end  train_n  train_events test_start   test_end  test_n  test_events  ae_threshold
    1  2019-10-18 2021-09-15      477            58 2021-09-30 2022-03-11     112           15       0.59712
    2  2019-10-18 2022-03-11      599            73 2022-03-28 2022-09-07     113           28       0.48117
    3  2019-10-18 2022-09-07      722           102 2022-09-22 2023-03-03     113            8       0.50082
    4  2019-10-18 2023-03-03      845           116 2023-03-20 2023-08-31     113            7       0.98009
    5  2019-10-18 2023-08-31      968           123 2023-09-15 2024-02-26     113            0       0.72633
    6  2019-10-18 2024-02-26     1091           123 2024-03-12 2024-08-22     112            0       0.57581
    7  2019-10-18 2024-08-22     1213           123 2024-09-06 2025-02-21     113            0       0.52303
    8  2019-10-18 2025-02-21     

In [3]:
# ============================================================
# Wyniki sieci rekurencyjnych
# ============================================================
y = oof['y_true'].astype(int)
common = oof.dropna(subset=['p_ens']).index
rows = []
for name, col in [('LSTM','p_lstm'), ('GRU','p_gru')]:
    s = oof.loc[common, col].dropna()
    yy = y.reindex(s.index).values
    half = len(s)//2
    thr = pl.select_threshold(yy[:half], s.values[:half]) if yy[:half].sum() else 0.5
    m = pl.evaluate(yy, s.values, thr); m['Model'] = name
    rows.append(m)
rnn = pd.DataFrame(rows).set_index('Model')
rnn.to_csv('results/tables/05_lstm_gru_comparison.csv')
print(rnn[['auc_roc','auc_pr','auc_pr_lift','f1','precision','recall','brier']].to_string())


       auc_roc  auc_pr  auc_pr_lift      f1  precision  recall   brier
Model                                                                 
LSTM    0.8365  0.3421       15.113  0.1685     0.0932  0.8824  0.0414
GRU     0.7535  0.3055       13.496  0.0949     0.0504  0.8235  0.0356
